# Wordle 任务介绍

在深入 SFT 概念之前，我们先来理解 Wordle 这个游戏——它是贯穿本教程和后续 RL 教程的核心任务。

---

## 游戏规则

Wordle 是一个经典的英文猜词游戏，规则简单但富有策略深度：

- **目标**：猜出一个隐藏的 5 字母英文单词。
- **回合限制**：最多猜 6 轮。
- **反馈机制**：每轮猜词后，系统对每个字母位置给出三种反馈：
  - **G（Green，绿色）**：字母正确，位置正确。
  - **Y（Yellow，黄色）**：字母正确，但位置不对。
  - **X（Gray，灰色）**：字母不在目标单词中。


```text
示例（目标词：SMILE）
第1轮：CRANE → C-r-a-n-e → X X X X X
        （5 个字母全部不在 SMILE 中）
第2轮：SLIME → S-l-i-m-e → G Y Y G G
        （S 和 M 位置对，L 和 I 在词中但位置不对，E 位置对）
第3轮：SMILE → s-m-i-l-e → G G G G G
        （全部正确，游戏结束！）
```


---

## 为什么选 Wordle 做训练任务？

Wordle 是一个理想的两阶段训练（SFT + RL）演示任务，原因如下：

### 1. 多轮决策的本质

Wordle 不是"输入→输出"的单步问题。模型需要根据每一轮的新反馈调整下一轮的猜测。这种**交互式决策**是单轮 SFT 无法直接优化的——因为"第 N 轮的最佳猜测"取决于前 N-1 轮的具体反馈，无法预先标注。

### 2. SFT 的明确定位

虽然 SFT 无法优化多轮决策策略，但 SFT 可以做一件关键的事：**让模型学会输出格式正确的猜测**。

基座模型（如 Qwen3-1.7B）不知道什么是 Wordle，不知道需要按 `<guess>WORD</guess>` 格式输出。SFT 通过已有的游戏对话数据教会模型这些"格式规范"，为后续 RL 阶段扫清障碍。

### 3. 清晰的评测指标

Wordle 提供了多层次的评测指标，可以清晰地量化 SFT 和 RL 各自的贡献：

| 指标 | 含义 | SFT 阶段 | RL 阶段 |
|------|------|----------|--------|
| `format_reward` | 是否按 `<guess>word</guess>` 格式输出 | **主要目标** | 保持 |
| `correct_answer` | 是否在 6 轮内猜对 | 基线 | **主要目标** |
| `partial_answer` | 部分字母/位置正确 | 基线 | **主要目标** |
| `length_bonus` | 猜对所用的轮数（越少越好） | 基线 | **主要目标** |

> **SFT 的核心贡献**：将 `format_reward` 从基座模型的 ~0.2 提升到 ~1.0（即模型始终按正确格式输出）。这是 RL 能够有效探索的前提。

---

## 数据格式

SFT 训练使用的是已有的 Wordle 游戏对话记录。每条数据包含：


```json
{
  "messages": [
    {"role": "system", "content": "You are a Wordle assistant..."},
    {"role": "user", "content": "Guess the word. G=correct, Y=wrong position, X=not in word."},
    {"role": "assistant", "content": "<guess>CRANE</guess>"},
    {"role": "user", "content": "Round 1: C-r-a-n-e → X X X X X"},
    {"role": "assistant", "content": "<guess>SLIME</guess>"},
    {"role": "user", "content": "Round 2: S-l-i-m-e → G Y Y G G"},
    {"role": "assistant", "content": "<guess>SMILE</guess>"},
    ...
  ],
  "answer": "SMILE"
}
```


每条数据是一个完整的多轮对话，`messages` 中 user 和 assistant 交替出现：user 给出游戏反馈（G/Y/X），assistant 给出猜测。`answer` 是隐藏的目标单词（仅用于评测，训练时不暴露给模型）。

---

## 训练数据的来源

我们使用 `willcb/V3-wordle` 数据集，来自 TextArena 项目。该数据集从英文词汇中筛选 5 字母名词，用随机策略生成完整游戏记录，约 2000 条训练样本。

> **数据质量说明**：这些游戏记录由随机策略生成，不是最优策略。SFT 阶段的目标不是让模型学会最优猜词策略（留给 RL），而是让它学会**格式规范**和**基本的 G/Y/X 反馈理解**。

---

## 小结

- Wordle 是一个 5 字母、6 轮、G/Y/X 反馈的猜词游戏。
- 两阶段训练方案：SFT 学格式，RL 学策略。
- SFT 的核心目标是将 `format_reward` 从 0.2 提升到 1.0。
- 训练数据是多轮对话格式，每条包含完整的 game play。

下一节，我们将深入 SFT 的核心概念，理解它是如何让模型学会输出正确格式的。

## 练习

1. （单选题）Wordle 游戏中，反馈 G 表示什么含义？
    A. 字母不在目标单词中
    B. 字母正确且位置正确
    C. 字母正确但位置错误
    D. 字母未被评估

2. （单选题）SFT 在 Wordle 两阶段训练中的核心目标是哪个评测指标？
    A. correct_answer
    B. partial_answer
    C. length_bonus
    D. format_reward

3. （判断题）Wordle 训练数据由随机策略生成而非最优策略，因为 SFT 的目标是学会格式规范而非最优猜词策略。


In [ ]:
!cat ./answer/01.02_answer.txt